In [1]:
import numpy as np
import pandas as pd
import statsmodels.tsa.stattools as ts
from pathlib import Path                    # Obsługa projektu
from scipy import stats
import statsmodels.api as sm
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# ── Konfiguracja 2: Wczytanie danych z 01_etl ──────────────────────

# Konfiguracja ścieżek (zgodna z pierwszym notebookiem)
PROJECT_ROOT = Path.cwd().parent  # ponieważ notebook jest w notebooks/
DATA_DIR = PROJECT_ROOT / "processed"  # uwaga: bez "data" w środku

OUT_DIR = PROJECT_ROOT / "outputs" / "html"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Odczyt danych
df = pd.read_parquet(DATA_DIR / "df.parquet")
df_rok = pd.read_parquet(DATA_DIR / "df_rok.parquet")

In [4]:
# ── SEKCJA 4: ANALIZA COUNTERFACTUAL ─────────────────────────────────────────



# ── KOLORY ────────────────────────────────────────────────────────────────────
CZERWONY   = '#C0392B'
NIEBIESKI  = '#1B3A6B'
ZLOTY      = '#C9982A'
ZIELONY    = '#1E8449'
TURKUSOWY  = '#1A7A6E'
MGLA       = '#8395A7'
SZARY_GRID = '#E4EAF2'
BG         = '#F7F9FC'
BIALY      = '#FFFFFF'

# ══════════════════════════════════════════════════════════════════════════════
# LOGIKA COUNTERFACTUAL
#
# Pytanie: jak wyglądałyby wpływy VAT gdyby reform NIE było?
#
# Metoda:
#   1. Model bazowy na 1999Q1–2016Q2 (przed JPK)
#   2. Ekstrapolacja rekurencyjna na 2016Q3–2025Q4
#   3. Luka = rzeczywistość − prognoza = efekt reform
#
# Model bazowy: AR(1) + AR(4) + sezonowość Q1/Q2/Q3 + konsumpcja + akcyza
# (JPK i COVID są zerowe w okresie bazowym, więc ich efekt mierzy luka)
#
# Dwie zmienne zależne:
#   A. vat_pkb_pct — efektywność poboru (pp)
#   B. vat_mld     — nominalne wpływy (mld PLN)
# ══════════════════════════════════════════════════════════════════════════════

# ── Podział próby ─────────────────────────────────────────────────────────────
# Indeks w formacie '1999Q1' (bez myślnika)
df_base = df[df.index <= '2016Q2'].copy()
df_proj = df[(df.index >= '2016Q3') & (df.index <= '2025Q4')].copy()

print('=' * 65)
print('  ANALIZA COUNTERFACTUAL')
print('=' * 65)
print(f'  Okres bazowy:    {df_base.index.min()}–{df_base.index.max()}'
      f'  (n={len(df_base)})')
print(f'  Okres projekcji: {df_proj.index.min()}–{df_proj.index.max()}'
      f'  (n={len(df_proj)})')

assert len(df_base) == 70, f'Oczekiwano 70, mam {len(df_base)}'
assert len(df_proj) == 38, f'Oczekiwano 38, mam {len(df_proj)}'
print('  ✅ Podział poprawny')

# ── Zmienne sezonowe ──────────────────────────────────────────────────────────
# endswith('Q1') działa dla obu formatów: '2016Q1' i '2016-Q1'
for d in [df_base, df_proj]:
    d['Q1'] = d.index.str.endswith('Q1').astype(int)
    d['Q2'] = d.index.str.endswith('Q2').astype(int)
    d['Q3'] = d.index.str.endswith('Q3').astype(int)

# ── Sprawdzenie zmiennych w okresie bazowym ───────────────────────────────────
print("\n  Kontrola zmiennych w okresie bazowym (1999Q1-2016Q2):")
print(f"    jpk_2016 - unikalne wartości: {df_base['jpk_2016'].unique()} (powinno być [0])")
print(f"    covid_2020 - unikalne wartości: {df_base['covid_2020'].unique()} (powinno być [0])")
print("  ✅ Zmienne poprawne - można estymować model")

# ══════════════════════════════════════════════════════════════════════════════
# HELPER — model counterfactual dla jednej zmiennej
# ══════════════════════════════════════════════════════════════════════════════

def counterfactual(df_base, df_proj, y_col, nazwa, jednostka):
    """
    Buduje model na okresie bazowym i prognozuje na okres reform.
    
    Model bazowy (okres przed reformami): 
        y = α + AR(1) + AR(4) + Q1+Q2+Q3
    
    UWAGA: JPK_2016 i COVID_2020 NIE wchodzą do modelu bazowego,
           bo w okresie 1999-2016 mają wariancję zero (same zera).
           Ich efekt jest mierzony przez LUKĘ = rzeczywistość - prognoza.
    """
    # ── Dane bazowe z lagami (BEZ JPK i COVID) ───────────────────────────────
    df_b = df_base[[y_col, 'Q1', 'Q2', 'Q3']].copy()
    df_b['lag1'] = df_b[y_col].shift(1)
    df_b['lag4'] = df_b[y_col].shift(4)
    df_b = df_b.dropna()

    y_b = df_b[y_col]
    X_b = sm.add_constant(df_b[['lag1', 'lag4', 'Q1', 'Q2', 'Q3']])

    model = sm.OLS(y_b, X_b).fit()

    print(f'\n  Model bazowy — {nazwa}')
    print(f'  n={int(model.nobs)}, '
          f'R²={model.rsquared:.4f}, '
          f'Adj.R²={model.rsquared_adj:.4f}')
    print(f'  Parametry:')
    print(f'    α={model.params["const"]:.4f}')
    print(f'    AR(1)={model.params["lag1"]:.4f}, AR(4)={model.params["lag4"]:.4f}')

    # ── Prognoza rekurencyjna ─────────────────────────────────────────────────
    # Potrzebujemy tylko zmiennych używanych w modelu
    df_all = pd.concat([
        df_base[[y_col, 'Q1', 'Q2', 'Q3']],
        df_proj[[y_col, 'Q1', 'Q2', 'Q3']],
    ])

    prognozy   = {}
    ci_lo_dict = {}
    ci_hi_dict = {}

    sigma = np.sqrt(model.mse_resid)
    idx_lista = df_all.index.tolist()

    for i, kwartal in enumerate(df_proj.index):
        pos     = idx_lista.index(kwartal)
        lag1_k  = idx_lista[pos - 1]
        lag4_k  = idx_lista[pos - 4]

        # Użyj prognozy jeśli już obliczona, inaczej rzeczywistej wartości
        lag1_v = (prognozy[lag1_k]
                  if lag1_k in prognozy
                  else df_all.loc[lag1_k, y_col])
        lag4_v = (prognozy[lag4_k]
                  if lag4_k in prognozy
                  else df_all.loc[lag4_k, y_col])

        # Pobierz wartości zmiennych egzogenicznych
        q1 = df_proj.loc[kwartal, 'Q1']
        q2 = df_proj.loc[kwartal, 'Q2']
        q3 = df_proj.loc[kwartal, 'Q3']

        X_p = np.array([1, lag1_v, lag4_v, q1, q2, q3])
        y_hat = float(model.params @ X_p)

        # CI rośnie z horyzontem prognozy
        sigma_h = sigma * np.sqrt(i + 1)

        prognozy[kwartal]   = y_hat
        ci_lo_dict[kwartal] = y_hat - 1.96 * sigma_h
        ci_hi_dict[kwartal] = y_hat + 1.96 * sigma_h

    y_counter = np.array([prognozy[k]   for k in df_proj.index])
    ci_lo     = np.array([ci_lo_dict[k] for k in df_proj.index])
    ci_hi     = np.array([ci_hi_dict[k] for k in df_proj.index])
    y_rzecz   = df_proj[y_col].values
    luka      = y_rzecz - y_counter

    # ── Wyniki ────────────────────────────────────────────────────────────────
    print(f'\n  Luka reform ({jednostka}):')
    print(f'  {"Kwartał":>10} {"Rzecz.":>10} {"Counter.":>10} '
          f'{"Luka":>10} {"Luka %":>8}')
    print('  ' + '-' * 52)

    for kw, r, c, l in zip(df_proj.index, y_rzecz, y_counter, luka):
        lp = l / c * 100 if c != 0 else 0
        print(f'  {kw:>10} {r:>10.3f} {c:>10.3f} '
              f'{l:>10.3f} {lp:>7.1f}%')

    luka_sr   = luka.mean()
    luka_skum = luka.sum() if jednostka == 'mld PLN' else None

    print(f'\n  Średnia luka kwartalna: {luka_sr:+.3f} {jednostka}')
    if luka_skum is not None:
        print(f'  Skumulowana 2016Q3–2025Q4: {luka_skum:+.1f} {jednostka}')

    return {
        'model':     model,
        'y_counter': y_counter,
        'ci_lo':     ci_lo,
        'ci_hi':     ci_hi,
        'luka':      luka,
        'luka_sr':   luka_sr,
        'luka_skum': luka_skum,
        'y_rzecz':   y_rzecz,
    }


# ══════════════════════════════════════════════════════════════════════════════
# COUNTERFACTUAL A — vat_pkb_pct
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '─' * 65)
print('  COUNTERFACTUAL A — vat_pkb_pct')
print('─' * 65)

wynik_pkb = counterfactual(
    df_base, df_proj,
    y_col='vat_pkb_pct',
    nazwa='vat_pkb_pct',
    jednostka='pp',
)

# ══════════════════════════════════════════════════════════════════════════════
# COUNTERFACTUAL B — vat_mld
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '─' * 65)
print('  COUNTERFACTUAL B — vat_mld')
print('─' * 65)

wynik_mld = counterfactual(
    df_base, df_proj,
    y_col='vat_mld',
    nazwa='vat_mld',
    jednostka='mld PLN',
)

# ══════════════════════════════════════════════════════════════════════════════
# PODSUMOWANIE
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 65)
print('  PODSUMOWANIE COUNTERFACTUAL')
print('=' * 65)
print(f'\n  Efektywność poboru (vat_pkb_pct):')
print(f'  Średnia luka kwartalna : {wynik_pkb["luka_sr"]:+.3f} pp')
print(f'  Roczny ekwiwalent      : {wynik_pkb["luka_sr"]*4:+.3f} pp')
print(f'\n  Nominalne wpływy (vat_mld):')
print(f'  Średnia luka kwartalna : {wynik_mld["luka_sr"]:+.1f} mld PLN')
print(f'  Skumulowana 2016Q3–2025Q4: {wynik_mld["luka_skum"]:+.1f} mld PLN')
print('=' * 65)


# ══════════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA — 4 panele
# ══════════════════════════════════════════════════════════════════════════════

lata_base = df_base.index.tolist()
lata_proj = df_proj.index.tolist()

# Format bez myślnika — zgodny z indeksem df
REFORMY = [
    ('2016Q3', NIEBIESKI, 'JPK (2016Q3)'),
    ('2018Q3', ZLOTY,     'Split payment (2018Q3)'),
    ('2019Q3', ZIELONY,   'Biała lista (2019Q3)'),
    ('2020Q2', MGLA,      'COVID-19 (2020Q2)'),
]

viz = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '① vat_pkb_pct — rzeczywistość vs counterfactual',
        '② vat_mld — rzeczywistość vs counterfactual',
        '③ Kwartalna luka reform (pp)',
        '④ Kwartalna luka reform (mld PLN)',
    ),
    vertical_spacing=0.20,
    horizontal_spacing=0.10,
)

# ── Panel 1: vat_pkb_pct ─────────────────────────────────────────────────────
viz.add_trace(go.Scatter(
    x=lata_base, y=df_base['vat_pkb_pct'].tolist(),
    mode='lines',
    name='Obserwowane (baza)',
    line=dict(color=MGLA, width=2, dash='dash'),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>VAT/PKB: %{y:.3f}%<extra></extra>',
), row=1, col=1)

viz.add_trace(go.Scatter(
    x=lata_proj, y=wynik_pkb['y_rzecz'].tolist(),
    mode='lines',
    name='Obserwowane (po JPK)',
    line=dict(color=CZERWONY, width=2.5),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>VAT/PKB rzecz.: %{y:.3f}%<extra></extra>',
), row=1, col=1)

viz.add_trace(go.Scatter(
    x=lata_proj + lata_proj[::-1],
    y=wynik_pkb['ci_hi'].tolist() + wynik_pkb['ci_lo'].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(131,149,167,0.15)',
    line=dict(color='rgba(0,0,0,0)'),
    name='95% CI',
    showlegend=True,
    hoverinfo='skip',
), row=1, col=1)

viz.add_trace(go.Scatter(
    x=lata_proj, y=wynik_pkb['y_counter'].tolist(),
    mode='lines',
    name='Counterfactual (bez reform)',
    line=dict(color=MGLA, width=2, dash='dot'),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>VAT/PKB cf: %{y:.3f}%<extra></extra>',
), row=1, col=1)

# ── Panel 2: vat_mld ─────────────────────────────────────────────────────────
viz.add_trace(go.Scatter(
    x=lata_base, y=df_base['vat_mld'].tolist(),
    mode='lines',
    name='VAT obserwowany (baza)',
    line=dict(color=MGLA, width=2, dash='dash'),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>VAT: %{y:.2f} mld<extra></extra>',
), row=1, col=2)

viz.add_trace(go.Scatter(
    x=lata_proj, y=wynik_mld['y_rzecz'].tolist(),
    mode='lines',
    name='VAT obserwowany (po JPK)',
    line=dict(color=CZERWONY, width=2.5),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>VAT rzecz.: %{y:.2f} mld<extra></extra>',
), row=1, col=2)

viz.add_trace(go.Scatter(
    x=lata_proj + lata_proj[::-1],
    y=wynik_mld['ci_hi'].tolist() + wynik_mld['ci_lo'].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(131,149,167,0.15)',
    line=dict(color='rgba(0,0,0,0)'),
    name='95% CI (mld)',
    showlegend=True,
    hoverinfo='skip',
), row=1, col=2)

viz.add_trace(go.Scatter(
    x=lata_proj, y=wynik_mld['y_counter'].tolist(),
    mode='lines',
    name='Counterfactual VAT (bez reform)',
    line=dict(color=MGLA, width=2, dash='dot'),
    showlegend=True,
    hovertemplate='<b>%{x}</b><br>VAT cf: %{y:.2f} mld<extra></extra>',
), row=1, col=2)

# ── Panel 3: Luka pp ─────────────────────────────────────────────────────────
kolory_luka_pkb = [
    CZERWONY if v > 0 else MGLA
    for v in wynik_pkb['luka']
]

viz.add_trace(go.Bar(
    x=lata_proj, y=wynik_pkb['luka'].tolist(),
    marker_color=kolory_luka_pkb,
    marker_line_color='white',
    marker_line_width=0.5,
    opacity=0.85,
    name='Luka (pp)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Luka: %{y:.3f} pp<extra></extra>',
), row=2, col=1)

viz.add_hline(y=0, row=2, col=1,
              line=dict(color='black', width=0.8))
viz.add_hline(
    y=wynik_pkb['luka_sr'], row=2, col=1,
    line=dict(color=CZERWONY, width=1.5, dash='dot'),
    annotation_text=f'Śr. {wynik_pkb["luka_sr"]:+.3f} pp',
    annotation_font=dict(size=9, color=NIEBIESKI),
    annotation_x=0.72
)

# ── Panel 4: Luka mld ────────────────────────────────────────────────────────
kolory_luka_mld = [
    CZERWONY if v > 0 else MGLA
    for v in wynik_mld['luka']
]

viz.add_trace(go.Bar(
    x=lata_proj, y=wynik_mld['luka'].tolist(),
    marker_color=kolory_luka_mld,
    marker_line_color='white',
    marker_line_width=0.5,
    opacity=0.98,
    name='Luka (mld PLN)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Luka: %{y:.1f} mld PLN<extra></extra>',
), row=2, col=2)

viz.add_hline(y=0, row=2, col=2,
              line=dict(color='black', width=0.8))
viz.add_hline(
    y=wynik_mld['luka_sr'], row=2, col=2,
    line=dict(color=NIEBIESKI, width=1.5, dash='dot'),
    annotation_text=f'Śr. {wynik_mld["luka_sr"]:+.1f} mld PLN',
    annotation_font=dict(size=9, color=NIEBIESKI),
    annotation_x=0.18,
)

# Linie reform na panelach luki
for panel_row, panel_col in [(2, 1), (2, 2)]:
    for kw, kolor, label in REFORMY:
        viz.add_shape(
            type='line',
            x0=kw, x1=kw,
            y0=0, y1=1,
            yref='paper',
            row=panel_row, col=panel_col,
            line=dict(color=kolor, width=1.2, dash='dash'),
        )
        viz.add_annotation(
            x=kw, y=0.995,
            xref=f'x{panel_row}{panel_col}',
            yref='paper',
            text=label.split('(')[0].strip(),
            font=dict(size=8, color=kolor),
            showarrow=False,
            xanchor='right',
            yshift=70,
            row=panel_row, col=panel_col,
        )

# ── LAYOUT ───────────────────────────────────────────────────────────────────
viz.update_layout(
    title=dict(
        text=(
            '<b>Analiza Counterfactual · Efekt reform podatkowych 2016–2025</b><br>'
            '<span style="font-size:11px;color:#8395A7">'
            f'Śr. luka VAT/PKB: {wynik_pkb["luka_sr"]:+.3f} pp · '
            f'Skumulowana luka VAT: {wynik_mld["luka_skum"]:+.1f} mld PLN<br>'
            'Opracowanie: Mateusz Durski · 2026'
            '</span>'
        ),
        x=0.01, xanchor='left',
        font=dict(size=15),
    ),
    height=1000,
    paper_bgcolor=BG,
    hovermode='x unified',
    template='simple_white',
    legend=dict(
        orientation='h',
        y=-0.15, x=0.5, xanchor='center',
        font=dict(size=9),
        bgcolor='rgba(247,249,252,0.9)',
        bordercolor=SZARY_GRID, borderwidth=1,
    ),
    margin=dict(t=120, b=90, l=60, r=60),
)

# ── OSIE ─────────────────────────────────────────────────────────────────────
for row, col, ylabel in [
    (1, 1, 'vat_pkb_pct (%)'),
    (1, 2, 'mld PLN'),
    (2, 1, 'Luka (pp)'),
    (2, 2, 'Luka (mld PLN)'),
]:
    viz.update_yaxes(title_text=ylabel, showgrid=True,
                     gridcolor=SZARY_GRID, row=row, col=col)
    viz.update_xaxes(title_text='Kwartał', tickangle=-45,
                     showgrid=False, row=row, col=col)

# ── EXPORT ───────────────────────────────────────────────────────────────────
file_path = OUT_DIR / "counterfactual.html"

viz.write_html(
    file_path,
    include_plotlyjs='cdn',
)
print('\n✅ Zapisano: counterfactual.html')
viz.show()

  ANALIZA COUNTERFACTUAL
  Okres bazowy:    1999Q1–2016Q2  (n=70)
  Okres projekcji: 2016Q3–2025Q4  (n=38)
  ✅ Podział poprawny

  Kontrola zmiennych w okresie bazowym (1999Q1-2016Q2):
    jpk_2016 - unikalne wartości: [0] (powinno być [0])
    covid_2020 - unikalne wartości: [0] (powinno być [0])
  ✅ Zmienne poprawne - można estymować model

─────────────────────────────────────────────────────────────────
  COUNTERFACTUAL A — vat_pkb_pct
─────────────────────────────────────────────────────────────────

  Model bazowy — vat_pkb_pct
  n=66, R²=0.4525, Adj.R²=0.4069
  Parametry:
    α=1.8775
    AR(1)=0.5618, AR(4)=0.1931

  Luka reform (pp):
     Kwartał     Rzecz.   Counter.       Luka   Luka %
  ----------------------------------------------------
      2016Q3      7.190      7.039      0.150     2.1%
      2016Q4      6.403      7.242     -0.839   -11.6%
      2017Q1      8.682      7.105      1.577    22.2%
      2017Q2      7.601      7.006      0.595     8.5%
      2017Q3      7